# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [9]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [10]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [12]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [14]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [15]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints data provided, tend to involve mismanagement and errors by loan servicers, such as:\n\n- Dealing with the lender or servicer, including receiving bad or incorrect information about the loan.\n- Errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n- Incorrect reporting of account status, such as being reported as delinquent when never late.\n- Struggling with how payments are handled, including difficulty applying extra payments to principal or paying off smaller loans.\n- Problems with loan transfer notifications and lack of transparency.\n- Disputes over interest rates, balances, or loan sale practices.\n\nOverall, a common issue appears to be the mishandling and miscommunication by loan servicers, leading to inaccurate account information, misapplied payments, and lack of transparency.'

In [16]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided data, yes, some complaints did not get handled in a timely manner. Specifically, one complaint from a user regarding their loan application status (Complaint ID: 12709087) was marked as "No" under "Timely response," indicating it was not handled promptly. The user reported that it has been over 15 days since they were told to wait 15 days for someone to contact them, and they had not heard back, with their loan still unprocessed.\n\nAdditionally, multiple complaints mention ongoing issues and delays in responses, cancellations, or unresolved problems despite passing the expected timelines. For example, a complaint from a user about an unresolved issue with their loan account (Complaint ID: 12973003) was handled "Yes" for response timeliness, but the issue persisted beyond the expected resolution period, indicating a delay in resolution.\n\nTherefore, at least some complaints, such as the one about the loan application not being processed timely, were not handled 

In [17]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons, including:\n\n1. **Accumulating interest and inability to afford increased payments:** Borrowers often faced interest that continued to grow, making it difficult to pay off the loans even after making payments for years. Lowering monthly payments or entering forbearance extended the repayment period and increased total debt due to interest accumulation.\n\n2. **Lack of clear communication and notification:** Many borrowers were not adequately informed about loan transfer details, when repayment was to resume, or changes in their loan status. This lack of communication led to missed payments and delinquencies.\n\n3. **Difficulty managing payments within financial hardships:** Borrowers' income levels and financial situations often made higher payments unaffordable, especially when interest kept increasing. Some felt they could not increase payments without compromising basic necessities.\n\n4. **Problems with loan servicers and

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [18]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [20]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [21]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the lender or servicer, particularly related to mismanagement, poor communication, or misleading information. Specific sub-issues include disputes over fees charged, trouble with how payments are applied (often to interest rather than principal), and receiving incorrect or unclear information about loan balances, terms, or qualification details. These issues highlight systemic problems in loan servicing practices, such as misinformation, lack of transparency, and difficulties in managing payments effectively.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided, all the complaints listed indicate that they were handled in a timely manner. Specifically, the complaints from rows 509, 288, 423, and 236 all state "Timely response?": "Yes." Therefore, there is no evidence in the provided data that any complaints were not handled in a timely manner.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues with the loan servicing process, miscommunication, and problems with payment plans. Some specific reasons highlighted are:\n\n- Being steered into the wrong types of forbearances, leading to increased loan amounts due to capitalized interest.\n- Lack of communication from loan servicers regarding important updates or decisions, such as transfer to new companies or changes in payment arrangements.\n- Technical issues or errors, such as payments being reversed or not processed correctly, which caused delays or defaults.\n- Failure of loan servicers to respond to requests for deferment or to provide necessary information, resulting in missed payments and negative impacts on credit scores.\n- Poor handling of automatic payments and not notifying borrowers about changes or issues, leading to unintentional default.\n- In some cases, borrowers experienced alleged deceptive practices, such as not being properly inform

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
####✅ Answer: What was the weather in Charlotte yesterday? Because BM25 works better when the query need retrival based on precise keyword matches  vs abstract and semantic generalization needs

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [24]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [25]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, a common issue with loans, specifically student loans, appears to be mishandling and misinformation by loan servicers. The most frequent problems include errors in loan balances, misapplied payments, wrongful denials of payment plans, lack of transparency in interest calculation, and inadequate communication or documentation. Additionally, borrowers frequently face issues related to incorrect or inconsistent loan information, unauthorized transfers of loans, and potential violations of privacy rights.\n\nIn summary, the most common issues reported are errors in loan balances and mismanagement by servicers, leading to confusion, incorrect billing, and credit reporting problems.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints did not get handled in a timely manner. For example, one complaint about loan account issues has been open for nearly 18 months without resolution, indicating a significant delay. Additionally, other complaints mention ongoing issues despite multiple follow-ups. However, in the specific case of the complaint summarized in the first document, the response was marked as "Yes" for being timely, but the issue remained unresolved after a long period. So, overall, there are instances where complaints experienced delays or were not resolved promptly.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a lack of clear information and communication about their loan obligations, increasing interest, and available repayment options. Many borrowers were unaware that they needed to repay their loans or did not receive adequate notifications or documentation from their lenders or servicers. Additionally, they faced difficulties with the complexity of interest accumulation, forbearance or deferment options that extended debt repayment periods and increased total interest, and challenges in managing loan payments alongside their financial circumstances. Some also experienced issues with incorrect or inconsistent account information, making it difficult to understand their actual balances or payment requirements. Overall, lack of transparency, inadequate guidance, and the accumulation of interest contributed to their inability to successfully repay their loans.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [29]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [30]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [31]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with loans appear to be related to mismanagement by servicers, errors in loan balances, improper handling of payments, and lack of transparency or communication. Many complaints highlight problems such as:\n\n- Errors in loan balances and interest calculations\n- Misapplied payments or inability to pay principal\n- Unauthorized transfer or reassignment of loans without proper notice\n- Wrongful reporting of delinquency or default\n- Failure to provide accurate information or guidance about repayment options\n- Disorganization and delays in resolving account issues\n- Unauthorized access or privacy violations\n- Problems with loan discharge, forgiveness, or cancellation\n\nOverall, a prevalent theme is that borrowers experience significant frustration and hardship due to poor management, lack of transparency, and errors by loan servicers and federal agencies.\n\nIf you need a specific summary, the most common issue is often the mish

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, several complaints indicate that issues did not get handled in a timely manner. For example:\n\n- Complaint #12709087 against MOHELA mentions the response was "Not timely" despite the company responding and closing with explanation.\n- Complaint #12973003 against EdFinancial Services states the response was "timely" but the issue persisted over 2-3 weeks with ongoing problems.\n- Multiple complaints, such as #13062402 and #13091395 against Nelnet and Maximus Federal Services, mention delays exceeding the expected response or resolution timeframes, with some noting that over a year has passed without resolution.\n- Several complaints specify that the company responses were "Closed with explanation" after delays, and in some cases, the consumer\'s issues remained unresolved.\n\nTherefore, the evidence suggests that some complaints were not handled in a timely manner.'

In [33]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as mismanagement by servicers, errors in loan balances, misapplied payments, wrongful denials of payment plans, and inadequate communication or guidance about repayment options. Additionally, some borrowers experienced financial hardships, lack of proper notice about delinquency or payment resumption, and unhelpful or misleading information from lenders or servicers. These factors contributed to difficulties in maintaining payments, incorrect reporting to credit bureaus, and in some cases, the unintended accumulation of interest or inability to access proper repayment plans, all of which hindered their ability to pay back their loans.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
###✅ Answer: I can't expect the user to be trained on prompting, and my app should handle queries written in all type of prose and loaded with errors. query reformulation would allow the retriver to retrive relevant information that semantically matches several reforumulations of the original query (ideally matching the actul intent of the query better). It increases retrival diversity, by retriving context that may come from lexically variant sources.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [34]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [35]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [36]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [37]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [38]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans seems to be related to errors or problems with loan servicing, such as incorrect information on credit reports, misapplied payments, wrongful denials of payment plans, and issues arising from the handling of loan balances and interest rates. Many complaints highlight systemic breakdowns, miscommunication, and discrepancies in loan accounts.\n\nIf I had to identify a pattern, it appears that **systemic errors, misreporting, and mismanagement by loan servicers** are the most frequently reported issues.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, several complaints mentioned delays or failure to receive timely responses from the companies. Specifically, the complaint related to the borrower waiting over multiple calls and extended wait times (up to seven hours) to get assistance indicates that some complaints were not handled in a timely manner. Additionally, the complaint regarding the dispute settlement waiting over 30 days without response suggests a delay in handling that issue.\n\nSo, yes, there are complaints recorded where responses or resolutions did not occur in a timely manner.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as financial hardship, lack of proper information, mismanagement by loan servicers, and misinformation about the terms and conditions of their loans. \n\nFor example, one complaint mentions that payments were resumed while the borrower was still attending college, without proper reevaluation based on the grace period, leading to delinquency. Another borrower struggled to repay due to severe financial hardship after graduating, relying on deferments and forbearances that increased the interest owed. Additionally, some borrowers were misled by educational institutions about the value and outcomes of their degrees, which impacted their ability to secure employment and repay their loans. \n\nOverall, difficulties in repayment often stem from inadequate communication, unexpected financial burdens, mismanagement, and misrepresentation by both educational institutions and loan servicing entities.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [44]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [45]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with student loans appear to involve dealing with lenders or servicers, such as errors in loan balances, misapplied payments, wrongful denials of repayment plans, and difficulties in obtaining accurate or clear information about the loans. Other frequent problems include receiving bad information about loan terms, incorrect reporting on credit reports, trouble with repayment plans or loan transfers, and issues related to mismanagement or mishandling of loan data.\n\nIn summary, the most common issue with loans tends to be problems related to loan servicing—specifically mismanagement, misinformation, and poor communication from loan servicers.'

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints explicitly mention delays or responses that were not handled in a timely manner. For example:\n\n- One complaint (Complaint ID: 12709087) indicates the response was **"No"** for timeliness.\n- Another complaint (Complaint ID: 13062402) states **"Yes"** for timeliness, implying it was handled within the expected timeframe.\n- Multiple complaints note delays, some exceeding the promised response window, or indicate that the issue remains unresolved for extended periods (e.g., complaints about months or over a year with no resolution or response).\n\nGiven this, it is clear that **some complaints did not get handled in a timely manner**. Specifically, complaints such as the one with Complaint ID 12709087 clearly state the response was untimely.\n\n**Therefore, the answer is:**  \n**Yes, some complaints did not get handled in a timely manner.**'

In [47]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to factors such as lack of adequate information about payment obligations, mismanagement or miscommunication from loan servicers, difficulties in navigating repayment plans, unforeseen financial hardships, and issues related to transferring or consolidating loans without proper notification. Many borrowers also faced challenges with accumulating interest due to forbearance or deferment options that did not stop interest from growing, making repayment more difficult over time. Additionally, some borrowers were unaware of their loan status or changes in servicing, leading to missed payments and credit report errors, which further hindered their ability to repay.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [49]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [50]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [51]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [52]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [53]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [54]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided complaints data, the most common issues with loans appear to be related to handling and servicing problems, including:\n\n- Struggling to repay or issues with repayment plans\n- Problems with loan forgiveness, cancellation, or discharge\n- Improper or illegal credit reporting and collection activities\n- Difficulties with loan account information, errors, or disputes\n- Poor communication from lenders or servicers\n- Problems related to loan account statuses, including defaults and reporting errors\n- Issues with borrower authorization, data breaches, or privacy violations\n\nOverall, the most prevalent issue seems to revolve around difficulties in communication, administrative errors, and questionable collection or reporting practices by loan servicers.\n\nIf I had to generalize, the most common issue with loans from this data is **servicing and administrative problems, including misreporting, communication failures, and difficulties with repayment or loan statu

In [55]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided data, it appears that several complaints were marked as responded to with the company\'s response being "Closed with explanation" and labeled as "Timely response: Yes." However, in these cases, there is no explicit mention that the complaints were handled in a fully satisfactory or timely manner from the complainants\' perspective. \n\nSpecifically, many complaints involve complex issues, disputes, or a lack of responsive follow-up, but the records indicate that the responses were considered timely by the entities involved. Without additional context or consumer feedback indicating unresolved issues or delays, it is difficult to definitively say that any complaints were not handled in a timely manner.\n\nTherefore, based on the available information, I would say that there is no clear evidence that any complaints were left unhandled in a timely manner.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People may fail to pay back their loans for various reasons, including difficulties in dealing with their lender or servicer, problems with payment processing, disputes over the legitimacy or status of their loans, and issues related to miscommunication or lack of transparency. For example, some borrowers experience trouble understanding their repayment terms, such as re-amortization after forbearance, or encounter delays and errors in payment processing. Others face complications due to incorrect or disputed account information, default status misreporting, or legal issues like invalid or voided debts. Additionally, some borrowers may feel overwhelmed by the stress, uncertainty, or perceived unfair practices by their loan servicers, which can impact their ability or willingness to repay.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅✅ Answer: If questions are repetitive, semantic chunking will merge the questions into similar chunks, losing distinction of questions, and providing redundant, blended, generic answers. One way is to not use semantic chunking for questions in FAQ, but alternatively, reduce the chunk size with clear seperators to presever the units witing each FAQ. Second option is to treat each questions as a seperate chunk. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE